# HJB Matching — Lensing Experiments

Train all 4 lensing variants (uncontrolled, concave, convex, flat) on Colab GPU.

Outputs are saved to `toy/outputs/lensing_<variant>/` for local figure generation.

In [ ]:
!git clone https://github.com/vishaal-krishnan/hjb_matching.git
%cd hjb_matching
!git checkout vishaal/refactor

In [ ]:
!pip install -q jax jaxlib dm-haiku optax pyyaml matplotlib

In [ ]:
import os
import pickle
import yaml
import numpy as np
import jax
import jax.numpy as jnp
import optax

from toy.distributions import DISTRIBUTIONS, nu_lensing, nu_convex, nu_flat
from toy.model import build_w_net
from toy.train import build_train_fns

NU_FUNCTIONS = {
    'concave': nu_lensing,
    'convex': nu_convex,
    'flat': nu_flat,
}

print(f"JAX devices: {jax.devices()}")

In [ ]:
def run_lensing(config_path, output_dir):
    """Train a lensing experiment, save weights + loss + full trajectory."""
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    os.makedirs(output_dir, exist_ok=True)

    N = cfg['N']
    key = jax.random.PRNGKey(0)

    init_sample = DISTRIBUTIONS[cfg['distribution']]
    if cfg.get('use_analytical_nu', False):
        nu_type = cfg.get('nu_type', 'concave')
        nu_fn = NU_FUNCTIONS[nu_type]
    else:
        nu_fn = None
    target_pos = jnp.array([cfg['target_pos']])

    w_net = build_w_net(cfg)
    key, subkey = jax.random.split(key)
    w_params = w_net.init(subkey, jnp.zeros((N, 2)), jnp.full((N,), 0, dtype=jnp.int32))

    w_opt = optax.adam(cfg['lr'])
    w_opt_state = w_opt.init(w_params)

    fns = build_train_fns(w_net, w_opt, cfg, nu_fn=nu_fn)
    train_step = fns['train_step']
    rollout = fns['rollout']

    loss_lst = []
    log_every = cfg.get('log_every', 50)

    for epoch in range(cfg['total_epochs']):
        key, subkey1, subkey2 = jax.random.split(key, 3)
        pos0 = init_sample(subkey1, N)

        w_params, w_opt_state, loss, loss1, loss2, loss3, final_pos, _ = train_step(
            w_params, w_opt_state, pos0, subkey2, target_pos
        )
        loss_lst.append(float(loss))

        if epoch % log_every == 0:
            print(f"  Epoch {epoch:4d}  loss={loss:.6f}")

    # Save a focusing trajectory for visualization
    key, k1, k2 = jax.random.split(key, 3)
    pos0 = init_sample(k1, N)
    _, _, traj_foc, _, _ = rollout(pos0, k2, w_params, None,
                                   "focusing", "pretraining", target_pos, 0)

    # Save outputs
    with open(f'{output_dir}/w_params.pkl', 'wb') as f:
        pickle.dump(w_params, f)
    with open(f'{output_dir}/loss_lst.pkl', 'wb') as f:
        pickle.dump(loss_lst, f)
    with open(f'{output_dir}/config_used.yaml', 'w') as f:
        yaml.dump(cfg, f)
    with open(f'{output_dir}/traj_focusing.pkl', 'wb') as f:
        pickle.dump(np.array(traj_foc), f)

    print(f"  Done. Final loss={loss_lst[-1]:.6f}. Saved to {output_dir}/")
    return w_params, loss_lst

## Run All 4 Variants

In [ ]:
print("=" * 50)
print("  Uncontrolled")
print("=" * 50)
run_lensing('toy/configs/lensing_uncontrolled.yaml', 'toy/outputs/lensing_uncontrolled')

In [ ]:
print("=" * 50)
print("  Concave Lens")
print("=" * 50)
run_lensing('toy/configs/lensing_concave.yaml', 'toy/outputs/lensing_concave')

In [ ]:
print("=" * 50)
print("  Convex Lens")
print("=" * 50)
run_lensing('toy/configs/lensing_convex.yaml', 'toy/outputs/lensing_convex')

In [ ]:
print("=" * 50)
print("  Flat Profile")
print("=" * 50)
run_lensing('toy/configs/lensing_flat.yaml', 'toy/outputs/lensing_flat')

## Download Outputs

In [ ]:
!zip -r lensing_outputs.zip toy/outputs/lensing_*/

try:
    from google.colab import files
    files.download('lensing_outputs.zip')
except ImportError:
    print("Not running on Colab. Find lensing_outputs.zip in the working directory.")